# 02 Tokenization and BPE

## Problem

为什么大模型不直接读取原始字符串？BPE 到底是什么，它到底在“合并”什么，为什么这种方法能成为现代 tokenizer 的核心思路之一？

## Dependencies

建议先完成 `01_python_and_matrix_foundations.ipynb`。这里默认你已经知道列表、字典、循环和最基础的统计直觉。


## Goals

- 说清楚 tokenizer 在整个 LLM 管线里的作用
- 区分 char-level、word-level、subword 三种切分思路的优缺点
- 理解 BPE 的训练过程和编码过程不是一回事
- 能用一个最小例子解释 BPE 为什么在压缩率和泛化之间取得平衡

## Scope and Approach

这一节不是一句“BPE 就是高频合并”就结束。我们会把问题拆成四层：

1. 模型为什么不能直接吃原始文本。
2. 为什么 char-level 和 word-level 各有明显问题。
3. BPE 的训练阶段到底学到了什么。
4. BPE 的编码阶段到底怎样把一个新词拆成子词。


## 为什么需要 tokenizer

模型本质上不会“读文字”，它只会处理数字。问题在于，原始文本并不是一串天然适合模型处理的数字序列，所以我们需要一个中间层，把文本变成离散的 token。

比如：

- 人看到的是：`newest`
- tokenizer 可能切成：`['new', 'est']`
- 模型真正吃进去的是：`[314, 982]`

所以 tokenizer 不是可有可无的小配件，而是模型输入管线的第一层接口。它决定了：

- 一段文本会被切成多长的序列
- 词表会有多大
- 新词、错拼、数字、代码、专有名词会不会很难处理


## 为什么不是字符，也不是整词

先看两个最朴素的方案。

### 1. Character level

优点：

- 词表很小
- 永远不会有未登录词

问题：

- 序列会非常长
- 一个常见词会被拆得太碎
- 模型必须跨很多步才能恢复词级语义

### 2. Word level

优点：

- 序列更短
- 常见词作为完整单位进入模型

问题：

- 词表可能爆炸
- 拼写变化、复数、时态、专有名词都会制造大量未登录词
- 对代码、多语言、噪声文本都不够稳

BPE 就落在这两者中间。它不是把所有东西拆到字符，也不要求所有词整词进词表，而是把高频片段逐步合成为更稳定的子词单位。


In [ ]:
from collections import Counter

# 一个非常小的训练语料。真实 tokenizer 会在极大规模语料上学习，
# 这里我们只保留最小例子，好让 merge 过程能一眼看清。
corpus = [
    'low low low lower newest widest',
    'low lower lowest',
    'new newer newest',
]

# 为了让“词结束了”这件事能被看见，BPE 教学实现里常给每个词加 </w>。
# 这不是为了神秘，而是为了区分：
#   'low' 结尾的 'w' 和词中间的 'w' 在统计上不完全一样。
def split_word(word):
    return tuple(list(word) + ['</w>'])

vocab = Counter()
for line in corpus:
    for word in line.split():
        vocab[split_word(word)] += 1

print('初始按字符拆开的词表:')
for symbols, freq in vocab.items():
    print(symbols, '->', freq)


## BPE 训练阶段到底学到了什么

这是最关键的一步：

**BPE 训练阶段学到的不是词义，而是一套 merge 规则。**

它会反复问一个问题：

- 现在所有词都拆成最小符号以后，哪个相邻 pair 出现得最多？
- 那就把这个 pair 合并成一个新符号。
- 合并之后，再重新统计新的 pair。
- 如此循环。

所以 BPE 更像一种从数据里学压缩规则的过程，而不是语言学解析器。


In [ ]:
def get_pair_stats(vocab_counter):
    # 统计所有相邻符号 pair 出现的频次。
    # 注意这里的统计是“带词频权重”的：某个词如果在语料里出现 10 次，
    # 那它内部的 pair 也应该被算 10 次。
    pair_counts = Counter()
    for symbols, freq in vocab_counter.items():
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += freq
    return pair_counts

def merge_pair(pair, vocab_counter):
    # 把某个最频繁 pair 合并成一个新符号。
    # 例如 ('l', 'o') -> 'lo'
    merged_vocab = Counter()
    a, b = pair
    merged_symbol = a + b

    for symbols, freq in vocab_counter.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            # 如果当前位置和下一个位置正好是要合并的 pair，
            # 就把它们折叠成一个更大的符号。
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        merged_vocab[tuple(new_symbols)] += freq
    return merged_vocab

def train_bpe(vocab_counter, num_merges=8):
    merges = []
    current_vocab = vocab_counter.copy()

    for step in range(num_merges):
        pair_stats = get_pair_stats(current_vocab)
        if not pair_stats:
            break

        best_pair, best_freq = pair_stats.most_common(1)[0]
        merges.append(best_pair)

        print(f'步骤 {step + 1}: 合并 {best_pair}, 频次 = {best_freq}')
        current_vocab = merge_pair(best_pair, current_vocab)

        print('合并后词表片段:')
        for symbols, freq in current_vocab.items():
            print('  ', symbols, '->', freq)
        print('-' * 40)

    return merges, current_vocab

merges, trained_vocab = train_bpe(vocab, num_merges=8)
print('学习到的 merge 规则 =', merges)


## 训练 merge 规则和编码新词不是一回事

第二个特别容易混淆的点是：

- **训练阶段**：从语料里学 merge 规则
- **编码阶段**：拿着学好的 merge 规则去切新文本

编码一个新词时，并不会重新训练 tokenizer。它只是按已经学好的 merge 优先级，尽可能把字符序列折叠成更大的子词单位。

所以当你问“BPE 是什么”时，最好分成两半回答：

- BPE 是一种学习 merge 规则的方法
- 同时也是一种用这些 merge 规则把新词编码成子词序列的方法


In [ ]:
def encode_word_with_bpe(word, merges):
    # 编码阶段从字符开始，但不会重新统计全语料，
    # 而是直接使用训练阶段已经学好的 merge 顺序。
    symbols = list(word) + ['</w>']
    merge_ranks = {pair: i for i, pair in enumerate(merges)}

    print('初始符号序列 =', symbols)
    while True:
        candidate_pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merge_ranks[pair], pair) for pair in candidate_pairs if pair in merge_ranks]

        # 如果当前序列里已经没有任何 pair 能匹配已学好的 merge 规则，
        # 编码过程就结束。
        if not ranked:
            break

        # rank 越小，表示这条 merge 规则越早学到，也越优先使用。
        _, best_pair = min(ranked)
        print('本轮使用 merge =', best_pair)

        a, b = best_pair
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        symbols = new_symbols
        print('合并后 =', symbols)

    return symbols

word = 'newest'
encoded = encode_word_with_bpe(word, merges)
print(f'最终编码: {word} -> {encoded}')

# 一个非常朴素的 decode：把 </w> 去掉再拼回去
print('decode ->', ''.join(token.replace('</w>', '') for token in encoded))


## BPE 为什么在工程上好用

BPE 的价值不在于它“完美”，而在于它做了一个非常实用的折中：

- 比 char-level 更短
- 比 word-level 更稳
- 对新词、拼写变化、代码片段有更好的泛化能力

但它也不是没有代价：

- merge 规则是统计学产物，不一定总符合人类直觉的词边界
- vocab size 太大时，embedding 参数会跟着变大
- vocab size 太小时，序列又会重新变长

所以 BPE 的核心不是“发明词义”，而是找到一个更适合大规模统计建模的离散表示方式。

## Common mistakes

- 误以为 token 一定等于单词。现代 tokenizer 很多时候处理的是子词片段。
- 只记得 BPE 在做“合并”，却没意识到它是在合并高频 pair，而不是随便拼接。
- 把训练 merge 规则和用 merge 规则编码新词混为一谈。
- 把更大的 vocab size 直接理解成一定更好，忽略了参数量、序列长度和泛化之间的平衡。

## Checkpoints

- 用自己的话回答：BPE 训练阶段到底学到了什么？
- 观察 `num_merges` 从 2 增加到 20 时，编码结果如何变化。
- 把训练语料换成中文短句，看看最频繁 pair 会不会不同。
- 解释为什么 BPE 对自然语言和代码都常常有效。

## Summary

BPE 本质上是一套从数据里学出来的 merge 规则。它通过逐步合并高频相邻符号，把纯字符序列压成更稳定的子词单位，从而在序列长度、词表大小和泛化能力之间找到一个工程上很有用的平衡点。

## Next Module

下一节进入 embedding。到那里，token 不再只是离散片段，而会第一次变成模型真正能计算的连续向量。
